# THERMOSCOPE · 02 Results
Reads the evidence files in `outputs/`.

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
R = json.load(open('../outputs/metrics_regression.json')); C = json.load(open('../outputs/metrics_classification.json'))
P = pd.read_csv('../outputs/predictions_test.csv'); json.load(open('../outputs/splits.json'))

## Degree selection (grouped CV, Z)

In [ ]:
pd.DataFrame(R['degree_selection'])

## Regression: grouped-CV and test performance

In [ ]:
rows = [dict(target=t, model=k, cv_rmse=v.get('cv'), **v['test']) for t, T in R['targets'].items() for k, v in T.items()]
pd.DataFrame(rows).set_index(['target', 'model']).round(4)

## Actual versus predicted and residuals (Ridge, Z)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].scatter(P.Z, P.pred_Z_ridge, c=P.phase, s=4); ax[0].plot([0, 1.2], [0, 1.2], 'k'); ax[0].set_xlabel('reference Z'); ax[0].set_ylabel('predicted Z')
ax[1].scatter(P.Pr, P.Z - P.pred_Z_ridge, c=P.phase, s=4); ax[1].set_xscale('log'); ax[1].set_xlabel('Pr'); ax[1].set_ylabel('residual')
ax[2].hist(P.Z - P.pred_Z_ridge, bins=50); ax[2].set_xlabel('residual')

## Per-fluid error

In [ ]:
P.assign(e=(P.Z - P.pred_Z_ridge) ** 2).groupby('fluid').e.mean().pow(0.5).rename('RMSE Z (ridge)')

## Classification

In [ ]:
for k in ('majority', 'lr', 'svm'):
    print(k, {m: round(C[k]['test'][m], 4) for m in ('acc', 'macroF1', 'balAcc')}); print(np.array(C[k]['test']['cm']))

## Hyper-parameter search

In [ ]:
cv = pd.read_csv('../outputs/cv_classification.csv'); cv[cv.model == 'svm_rbf'].pivot(index='C', columns='gamma', values='cv_macro_f1').round(4)

In [ ]:
cr = pd.read_csv('../outputs/cv_regression.csv'); cr[(cr.target == 'Z') & (cr.model == 'ridge')].pivot(index='penalty', columns='degree', values='cv_rmse').round(4)

## Gram matrix diagnostics and ablations

In [ ]:
pd.read_csv('../outputs/gram_diagnostics.csv')

In [ ]:
json.load(open('../outputs/ablations.json'))